# projectv2: Is the gender-misalignment direction a composition?

Runs the full redesigned experiment (see `projectv2/README.md`) on a Colab **A100 80GB** runtime.

The pipeline is fully resumable — if Colab disconnects, rerun the pipeline cell with the same `RUN_NAME` and it continues from the last completed stage (judging stages checkpoint after every metric).

**Before running:** Runtime → Change runtime type → A100 GPU.

In [ ]:
# 1. Verify GPU
!nvidia-smi

In [ ]:
# 2. Clone the repo (EDIT THE URL to your GitHub repo) and install dependencies
import os

REPO_URL = "https://github.com/YOUR_USERNAME/sexist_misalignment.git"  # <-- EDIT
REPO_DIR = "/content/sexist_misalignment"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

# Colab preinstalls an old torchao (0.10) that makes peft's LoRA dispatcher
# raise ImportError ("only versions above 0.16.0 are supported"). Nothing in
# this pipeline uses torchao, so remove it.
%pip uninstall -q -y torchao
%pip install -q -r projectv2/requirements.txt

In [ ]:
# 3. Hugging Face login (needed to download the models)
#    Store your token under Colab secrets as HF_TOKEN, or paste it when prompted.
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret HF_TOKEN")
except Exception:
    login()  # interactive prompt

In [ ]:
# 4. (Optional but recommended) persist artifacts to Google Drive so a runtime
#    recycle does not lose completed stages. Skip this cell to keep outputs local.
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/projectv2_outputs
!rm -rf outputs_v2 && ln -s /content/drive/MyDrive/projectv2_outputs outputs_v2
print("outputs_v2 -> Google Drive")

In [ ]:
# 5. RUN THE FULL PIPELINE (all data, statistics, plots, report)
#    Use configs/debug.yaml first for a ~30 min end-to-end smoke test.
#    Full standard run is ~6-10 h; rerun this cell to resume after a disconnect.

CONFIG = "projectv2/configs/standard.yaml"   # or projectv2/configs/debug.yaml
RUN_NAME = "colab_standard_v1"               # keep fixed to enable resume

!python -m projectv2.run_all --config {CONFIG} --run-name {RUN_NAME}

In [ ]:
# 6. View the report and plots inline
from pathlib import Path
from IPython.display import Image, Markdown, display

run_dir = Path("outputs_v2") / RUN_NAME
display(Markdown((run_dir / "summary.md").read_text(encoding="utf-8")))
for png in sorted((run_dir / "plots").glob("*.png")):
    print(png.name)
    display(Image(str(png)))